In [1]:
# # Install dependencies for Unsloth + GPT-OSS
# !pip install --upgrade -qqq uv
# !uv pip install -qqq \
#     "torch>=2.8.0" "triton>=3.4.0" numpy pillow torchvision bitsandbytes "transformers==4.56.2" \
#     "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
#     "unsloth[base] @ git+https://github.com/unslothai/unsloth" \
#     git+https://github.com/triton-lang/triton.git@0add68262ab0a2e33b84524346cb27cbb2787356#subdirectory=python/triton_kernels
# !uv pip install --upgrade --no-deps transformers==4.56.2 tokenizers trl==0.22.2 unsloth unsloth_zoo

# # Install openai_harmony (Harmony protocol tools)
# !pip install -q openai-harmony jupyter_client pandas datasets


In [2]:
class CONFIG:
    TRAIN_SIZE = 50_000
    BATCH_SIZE = 32 
    EPOCHS = 1 
    LEARNING_RATE = 2e-4
    MAX_SEQ_LENGTH = 4096
    MODEL_PATH = None 
    KAGGLE=True

cfg = CONFIG()
cfg.MODEL_PATH = "/kaggle/input/gpt-oss-20b-bnb-4bit/transformers/unsloth/1" if cfg.KAGGLE else "unsloth/gpt-oss-20b"

In [3]:
!python --version

Python 3.12.12


In [4]:
print("STARTING THE AHHHHHHHHHHHHHHHHHHHHHHHHHHHH")

STARTING THE AHHHHHHHHHHHHHHHHHHHHHHHHHHHH


In [5]:
if cfg.KAGGLE:
    !uv pip install --system --no-index --find-links='/kaggle/input/unsloth-py-3-12/unsloth' 'unsloth'


Using Python 3.12.12 environment at: /usr
Resolved 86 packages in 325ms
Prepared 29 packages in 14.47s
Uninstalled 21 packages in 1.70s
Installed 29 packages in 8.87s
 + bitsandbytes==0.49.1
 + cut-cross-entropy==25.1.1
 - datasets==4.4.2
 + datasets==4.3.0
 - fsspec==2025.10.0
 + fsspec==2025.9.0
 + msgspec==0.20.0
 - multiprocess==0.70.18
 + multiprocess==0.70.16
 - nvidia-cublas-cu12==12.6.4.1
 + nvidia-cublas-cu12==12.8.4.1
 - nvidia-cuda-cupti-cu12==12.6.80
 + nvidia-cuda-cupti-cu12==12.8.90
 - nvidia-cuda-nvrtc-cu12==12.6.77
 + nvidia-cuda-nvrtc-cu12==12.8.93
 - nvidia-cuda-runtime-cu12==12.6.77
 + nvidia-cuda-runtime-cu12==12.8.90
 - nvidia-cufft-cu12==11.3.0.4
 + nvidia-cufft-cu12==11.3.3.83
 - nvidia-cufile-cu12==1.11.1.6
 + nvidia-cufile-cu12==1.13.1.3
 - nvidia-curand-cu12==10.3.7.77
 + nvidia-curand-cu12==10.3.9.90
 - nvidia-cusolver-cu12==11.7.1.2
 + nvidia-cusolver-cu12==11.7.3.90
 - nvidia-cusparse-cu12==12.5.4.2
 + nvidia-cusparse-cu12==12.5.8.93
 - nvidia-nccl-cu12==2.

In [6]:
try:
    from unsloth import FastLanguageModel
except:
    !uv pip install --system --no-index --find-links='//kaggle/input/unsloth-library/unsloth' 'unsloth'
    from unsloth import FastLanguageModel

    

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


2026-01-16 13:32:17.126262: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1768570337.475848      44 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1768570337.583462      44 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1768570338.440533      44 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768570338.440563      44 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768570338.440566      44 computation_placer.cc:177] computation placer alr

🦥 Unsloth Zoo will now patch everything to make training faster!


In [7]:
local_files_only = True if cfg.KAGGLE else False

In [8]:
# del model
# del tokenizer

In [9]:
import torch
dtype = None

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = cfg.MODEL_PATH ,
    dtype = dtype, # None for auto detection
    max_seq_length = cfg.MAX_SEQ_LENGTH, # Choose any for long context!
    load_in_4bit = True,  # 4 bit quantization to reduce memory
    full_finetuning = False, # [NEW!] We have full finetuning now!
    local_files_only = local_files_only
)

==((====))==  Unsloth 2026.1.3: Fast Gpt_Oss patching. Transformers: 4.57.1.
   \\   /|    NVIDIA H100 80GB HBM3. Num GPUs = 1. Max memory: 79.437 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.1+cu128. CUDA: 9.0. CUDA Toolkit: 12.8. Triton: 3.5.1
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [10]:
import pandas as pd
if not cfg.KAGGLE:
    import kagglehub
    from kagglehub import KaggleDatasetAdapter

    # Set the path to the file you'd like to load
    file_path = "filtered_low_pass_1_or_2.jsonl"

    # Load the latest version
    df = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "barnobarno/nemotron-low-reasoning-pass-rate-1-2",
    file_path,
    # Provide any additional arguments like 
    # sql_query or pandas_kwargs. See the 
    # documenation for more information:
    # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpandas
    )
if cfg.KAGGLE:
    df = pd.read_json("/kaggle/input/nemotron-low-reasoning-pass-rate-1-2/filtered_low_pass_1_or_2.jsonl", lines=True)



In [11]:
from datasets import Dataset

train_data = df.copy()
train_data.drop(columns=["uuid","original_expected_answer","license" ,"used_in" ,"user_name" ,"user_url" ,"url"], inplace=True ,axis=1)
#train_data.dropna(inplace=True)
train_data = train_data[train_data["tools"].isna()]
train_data.drop(columns=["tools"], inplace=True ,axis=1)
def remove_none_keys(messages):
    return [{k: v for k, v in entry.items() if v is not None} for entry in messages]
def format_for_gpt_oss(example):
    messages = example['messages']
    new_messages = []
    
    for msg in messages:
        new_msg = msg.copy()
        
        # 1. Rename 'reasoning_content' to 'thinking'
        if 'reasoning_content' in new_msg:
            new_msg['thinking'] = new_msg.pop('reasoning_content')
        
        # 2. Ensure intermediate tool steps don't conflict
        # The template raises an error if you have BOTH 'thinking' and 'content' 
        # inside a tool call message. Your data has content='', which is fine, 
        # but purely safe practice is to ensure it is None or empty.
        if new_msg.get('tool_calls') and new_msg.get('thinking'):
             new_msg['content'] = "" # Ensure this is empty to avoid template error

        new_messages.append(new_msg)
    
    return {'messages': new_messages}

# Apply to your dataset
train_data_formatted = train_data.apply(format_for_gpt_oss, axis=1)
train_data["messages"] = train_data_formatted
train_data["messages"].iloc[0]["messages"]
train_data["AA"] = train_data["messages"].apply(lambda x: x["messages"])


# 1. Shuffle first (Optional but recommended)
# This ensures you don't just pick the "first" solution for a problem if there are duplicates.
train_data_shuffled = train_data.copy()  #  .sample(frac=1, random_state=42)

# 2. Get the pool of unique problems
# This drops all duplicates, leaving you with exactly 590k rows (1 per unique answer)
unique_pool = train_data_shuffled.drop_duplicates(subset=["expected_answer"])
print(f"Unique Shape: {unique_pool.shape}")

# 3. Sample your target 100k from that pool
dataset = unique_pool.sample(n=cfg.TRAIN_SIZE, random_state=42)

# Both should be 100,000


#dataset = train_data.iloc[0:cfg.TRAIN_SIZE].copy()

dataset["AA"] = dataset["AA"].apply(remove_none_keys)

dataset["text"] = dataset.apply(lambda row: tokenizer.apply_chat_template(
    row["AA"], 
    tokenize=False, 
    add_generation_prompt=True,
    reasoning_effort="low"
), axis=1)
print(f"Final dataset shape: {dataset.shape}")
print(f"Unique answers: {dataset['expected_answer'].nunique()}")


Unique Shape: (52297, 7)
Final dataset shape: (50000, 8)
Unique answers: 50000


In [12]:
train_data.head()

,problem,expected_answer,changed_answer_to_majority,data_source,messages,metadata,AA
0,"Let \(a, b, c, d\) be four positive integers. ...",25,True,aops,"{'messages': [{'role': 'user', 'content': 'Sol...","{'reason_low_with_tool': {'count': 8, 'pass': ...","[{'role': 'user', 'content': 'Solve the follow..."
1,Find all real solutions to the system:\n\[ x^3...,"(x,y,z)=\bigl(2\cos\theta,\;2\cos3\theta,\;2\c...",True,aops,"{'messages': [{'role': 'user', 'content': 'Sol...","{'reason_low_with_tool': {'count': 8, 'pass': ...","[{'role': 'user', 'content': 'Solve the follow..."
2,What is the total number of squares that can b...,\;\n\begin{cases}\n\displaystyle\frac{m(m+1)(3...,True,aops,"{'messages': [{'role': 'user', 'content': 'Sol...","{'reason_low_with_tool': {'count': 8, 'pass': ...","[{'role': 'user', 'content': 'Solve the follow..."
3,Given the function \( f(x) = \frac{2x}{1 + x^2...,\;0\le\displaystyle\sum_{k=1}^{n}f^{-1}(x_{k})...,True,aops,"{'messages': [{'role': 'user', 'content': 'Sol...","{'reason_low_with_tool': {'count': 8, 'pass': ...","[{'role': 'user', 'content': 'Solve the follow..."
4,"In triangle \(ABC\), the altitude, angle bisec...",22.5^{\circ},True,aops,"{'messages': [{'role': 'user', 'content': 'Sol...","{'reason_low_with_tool': {'count': 8, 'pass': ...","[{'role': 'user', 'content': 'Solve the follow..."


In [13]:
len(train_data[train_data["changed_answer_to_majority"]==False])

68061

In [14]:
# Fastest way to get the COUNT of unique items
count = train_data["problem"].nunique()
print(f"Unique items: {count}")

Unique items: 59545


In [15]:
# from datasets import Dataset

# # 1. Pre-processing
# train_data = df.copy()
# train_data.drop(columns=["uuid", "original_expected_answer", "license", "used_in", "user_name", "user_url", "url"], inplace=True, axis=1)

# # Filter out rows with tools (as per your snippet)
# train_data = train_data[train_data["tools"].isna()]
# train_data.drop(columns=["tools"], inplace=True, axis=1)

# def remove_none_keys(messages):
#     return [{k: v for k, v in entry.items() if v is not None} for entry in messages]

# def format_for_gpt_oss(example):
#     messages = example['messages']
    
#     # --- MODIFICATION START ---
#     # Initialize with the System Prompt
#     new_messages = [{
#         "role": "system", 
#         "content": "YOU ARE A MATH EXPERT"
#     }]
#     # --- MODIFICATION END ---
    
#     for msg in messages:
#         # Optional: Skip existing system prompts to strictly enforce your new one
#         if msg.get('role') == 'system':
#             continue

#         new_msg = msg.copy()
        
#         # 1. Rename 'reasoning_content' to 'thinking'
#         if 'reasoning_content' in new_msg:
#             new_msg['thinking'] = new_msg.pop('reasoning_content')
        
#         # 2. Ensure intermediate tool steps don't conflict
#         if new_msg.get('tool_calls') and new_msg.get('thinking'):
#              new_msg['content'] = "" 

#         new_messages.append(new_msg)
    
#     return {'messages': new_messages}

# # Apply to your dataset
# train_data_formatted = train_data.apply(format_for_gpt_oss, axis=1)
# train_data["messages"] = train_data_formatted

# # Extract the list of messages
# train_data["AA"] = train_data["messages"].apply(lambda x: x["messages"])

# # Create the final dataset slice
# dataset = train_data.iloc[0:cfg.TRAIN_SIZE].copy()
# dataset["AA"] = dataset["AA"].apply(remove_none_keys)

# # Apply Chat Template
# dataset["text"] = dataset.apply(lambda row: tokenizer.apply_chat_template(
#     row["AA"], 
#     tokenize=False, 
#     add_generation_prompt=True,
#     reasoning_effort="low"
# ), axis=1)

In [16]:
dataset["text"].iloc[-1]

"<|start|>system<|message|>You are ChatGPT, a large language model trained by OpenAI.\nKnowledge cutoff: 2024-06\nCurrent date: 2026-01-16\n\nReasoning: low\n\n# Valid channels: analysis, commentary, final. Channel must be included for every message.\nCalls to these tools must go to the commentary channel: 'functions'.<|end|><|start|>user<|message|>Solve the following math problem. Make sure to put the answer (and only answer) inside \\boxed{}.\n\nFind the generator matrix for a $\\left[4,2,2\\right]$ linear code.<|end|><|start|>assistant<|channel|>final<|message|>**Step 1 – Understand the parameters**  \n\nA linear code with parameters \\([n,k,d]=[4,2,2]\\) is a binary (or \\(q\\)-ary) code of length \\(n=4\\), dimension \\(k=2\\) and minimum Hamming distance \\(d=2\\).  \nSince \\(k=2\\), the code space is a 2‑dimensional subspace of \\(\\mathbb{F}_2^{4}\\).  \nThe generator matrix \\(G\\) must be a \\(2\\times 4\\) matrix whose rows are a basis of that subspace, and it must generate

In [17]:
# Add LoRA adapters with rank 16
hf_dataset = Dataset.from_pandas(dataset)
model = FastLanguageModel.get_peft_model(
    model,
    r = 128,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 256,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

Unsloth: Making `model.base_model.model.model` require gradients


In [18]:
import gc
import torch
gc.collect()


0

In [19]:
from trl import SFTConfig, SFTTrainer
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=hf_dataset,
    dataset_text_field="text",
    max_seq_length=cfg.MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    args=SFTConfig(
        per_device_train_batch_size=cfg.BATCH_SIZE,
        gradient_accumulation_steps=2,
        warmup_steps=5,
        num_train_epochs=1, 
       # max_steps=10 ,
        learning_rate=cfg.LEARNING_RATE,
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
        report_to="none",
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        group_by_length=True
    ),
)

gpt_oss_kwargs = dict(
    instruction_part="<|start|>user<|message|>", 
    response_part="<|start|>assistant<|channel|>final<|message|>"
)

trainer = train_on_responses_only(
    trainer,
    **gpt_oss_kwargs,
)
print("TRAINING READY")

Unsloth: Tokenizing ["text"] (num_proc=30):   0%|          | 0/50000 [00:00<?, ? examples/s]

Map (num_proc=30):   0%|          | 0/50000 [00:00<?, ? examples/s]

TRAINING READY


In [20]:
# 1. Train the model
trainer_stats = trainer.train()

# 2. Memory stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)

# 3. Time stats
# 'train_runtime' is in seconds
train_time_seconds = trainer_stats.metrics.get('train_runtime', 0)
train_time_minutes = round(train_time_seconds / 60, 2)

print(f"Peak reserved memory = {used_memory} GB")
print(f"Total training time  = {train_time_minutes} minutes ({train_time_seconds:.2f} seconds)")

The model is already on multiple devices. Skipping the move to device specified in `args`.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 50,000 | Num Epochs = 1 | Total steps = 782
O^O/ \_/ \    Batch size per device = 32 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (32 x 2 x 1) = 64
 "-____-"     Trainable parameters = 63,700,992 of 20,978,458,176 (0.30% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,0.498000
2,0.501200
3,0.431500
4,0.445700
5,0.429100
6,0.398200
7,0.407200
8,0.376800
9,0.382900
10,0.375500


Peak reserved memory = 76.438 GB
Total training time  = 104.38 minutes (6263.06 seconds)


In [21]:
model.save_pretrained("gpt_oss_20b_nemotronv2_low")
tokenizer.save_pretrained("gpt_oss_20b_nemotronv2_low")
print("Model saved to 'gpt_oss_20b_nemotronv2_low'")

Model saved to 'gpt_oss_20b_nemotronv2_low'
